https://judge.nitro-ai.org/competitions/nitro/rise-2026-open-qualifier-1/2/view

In [2]:
import os
import glob
import random
import numpy as np
from PIL import Image

from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.decomposition import PCA

In [5]:
DATASET_ROOT = "dataset/dataset"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

BATCH_SIZE = 64
EMBED_DIM = 128
EPOCHS = 25
LR = 1e-4
IMAGE_SIZE = 224
MARGIN = 0.3
DEVICE = 'cuda'

# os.makedirs("checkpoints", exist_ok=True)
# os.makedirs("submissions", exist_ok=True)

In [6]:
all_murals = sorted(os.listdir(TRAIN_DIR))
random.shuffle(all_murals)

val_murals = all_murals[:50]
train_murals = all_murals[50:]

print(f"Train murals: {len(train_murals)}")
print(f"Val murals: {len(val_murals)}")

Train murals: 450
Val murals: 50


In [14]:
train_transform = transforms.Compose([

    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.7, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.5,
        scale=(0.02, 0.25),
        ratio=(0.3, 3.3),
        value=0
    ),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [15]:
class TripletSequenceDataset(Dataset):

    def __init__(self, root_dir, mural_list, transform=None):
        self.root_dir = root_dir
        self.mural_list = mural_list
        self.transform = transform

        self.sequences = []

        for mural in mural_list:
            mural_path = os.path.join(root_dir, mural)

            images = sorted(glob.glob(os.path.join(mural_path, "*.jpg")))

            if len(images) < 3:
                continue

            self.sequences.append(images)

    def __len__(self):
        return 50000

    def load_image(self, path):
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img

    def __getitem__(self, idx):

        seq = random.choice(self.sequences)

        n = len(seq)

        # anchor index
        a_idx = random.randint(1, n - 2)

        # positive = nearby frame
        pos_candidates = []

        if a_idx - 1 >= 0:
            pos_candidates.append(a_idx - 1)

        if a_idx + 1 < n:
            pos_candidates.append(a_idx + 1)

        p_idx = random.choice(pos_candidates)

        # negative = far away frame
        neg_candidates = [
            i for i in range(n)
            if abs(i - a_idx) > max(2, n // 4)
        ]

        if len(neg_candidates) == 0:
            neg_candidates = [
                i for i in range(n)
                if i != a_idx and i != p_idx
            ]

        n_idx = random.choice(neg_candidates)

        anchor = self.load_image(seq[a_idx])
        positive = self.load_image(seq[p_idx])
        negative = self.load_image(seq[n_idx])

        return anchor, positive, negative

In [16]:
class EmbeddingNet(nn.Module):

    def __init__(self, embed_dim=128):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        in_features = backbone.fc.in_features

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.embedding = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, embed_dim)
        )

    def forward(self, x):

        x = self.backbone(x)

        x = self.embedding(x)

        x = nn.functional.normalize(x, p=2, dim=1)

        return x

In [17]:
train_dataset = TripletSequenceDataset(
    TRAIN_DIR,
    train_murals,
    train_transform
)

val_dataset = TripletSequenceDataset(
    TRAIN_DIR,
    val_murals,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model = EmbeddingNet(EMBED_DIM).to(DEVICE)

criterion = nn.TripletMarginLoss(
    margin=MARGIN,
    p=2
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [18]:
best_val = 999999

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0
    pbar = tqdm(train_loader)

    for anchor, positive, negative in pbar:

        anchor = anchor.to(DEVICE)
        positive = positive.to(DEVICE)
        negative = negative.to(DEVICE)

        optimizer.zero_grad()

        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        pbar.set_description(
            f"Epoch {epoch+1} Train Loss: {loss.item():.4f}"
        )

    train_loss /= len(train_loader)

    # ---------------- VAL ----------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for anchor, positive, negative in val_loader:

            anchor = anchor.to(DEVICE)
            positive = positive.to(DEVICE)
            negative = negative.to(DEVICE)

            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)

            loss = criterion(emb_a, emb_p, emb_n)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step()

    print()
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    # save best
    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            "checkpoints/best_model.pth"
        )

        print("Saved best model")

Epoch 1 Train Loss: 0.0964:   2%|▏         | 15/782 [00:28<24:21,  1.90s/it]


KeyboardInterrupt: 

In [ ]:
print("\nRunning inference...")

model.load_state_dict(
    torch.load("checkpoints/best_model.pth")
)

model.eval()

submission_lines = []

@torch.no_grad()
def extract_embedding(img_path):

    img = Image.open(img_path).convert("RGB")

    img = val_transform(img)

    img = img.unsqueeze(0).to(DEVICE)

    emb = model(img)

    return emb.squeeze(0).cpu().numpy()

test_folders = sorted(os.listdir(TEST_DIR))

for folder in tqdm(test_folders):

    folder_path = os.path.join(TEST_DIR, folder)

    images = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))

    embeddings = []

    for img_path in images:

        emb = extract_embedding(img_path)

        embeddings.append(emb)

    embeddings = np.stack(embeddings)

    # ========================================================
    # PCA TO 1D
    # ========================================================

    pca = PCA(n_components=1)

    coords = pca.fit_transform(embeddings).squeeze()

    # sort by coordinate
    sorted_indices = np.argsort(coords)

    ordered_images = [images[i] for i in sorted_indices]

    # save submission lines
    for rank, img_path in enumerate(ordered_images):

        filename = os.path.basename(img_path)

        submission_lines.append(
            f"{folder},{filename},{rank}"
        )
